#### **Imports & Setup**

In [1]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Styling for professional charts
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

#### **Load Raw D365 Tables**

In [5]:
DATA_PATH = "../data/raw" 

item_master = pd.read_csv(f"{DATA_PATH}/d365_item_master.csv")
sales_lines = pd.read_csv(f"{DATA_PATH}/d365_sales_lines.csv")
item_ledgers = pd.read_csv(f"{DATA_PATH}/d365_item_ledger_entries.csv")
warehouse_entries = pd.read_csv(f"{DATA_PATH}/d365_warehouse_entries.csv")
freight_invoices = pd.read_csv(f"{DATA_PATH}/carrier_freight_invoices.csv") # Updated file name

print("--- SPORTLUX D365 Dataset Shapes ---")
print(f"Item Master: {item_master.shape}")
print(f"Sales Lines: {sales_lines.shape}")
print(f"Item Ledgers: {item_ledgers.shape}")
print(f"Warehouse Entries: {warehouse_entries.shape}")
print(f"Freight Invoices: {freight_invoices.shape}")

--- SPORTLUX D365 Dataset Shapes ---
Item Master: (180, 11)
Sales Lines: (1000, 19)
Item Ledgers: (1000, 9)
Warehouse Entries: (1000, 8)
Freight Invoices: (1000, 12)


#### **Master Data Quality & Missing Value Audit**

In [6]:
# Check for missing HS codes and posting groups (critical for D365 pre-distribution audit)
print("--- Item Master Missing Values Audit ---")
missing_summary = item_master.isnull().sum()
print(missing_summary[missing_summary > 0])

# Calculate percentage of items missing HS Codes and Posting Groups
total_items = len(item_master)
missing_hs_count = item_master['HS_Code'].isnull().sum()
missing_pg_count = item_master['Gen_Prod_Posting_Group'].isnull().sum()

missing_hs_pct = (missing_hs_count / total_items) * 100
missing_pg_pct = (missing_pg_count / total_items) * 100

print(f"\nAudit Alert: {missing_hs_count} SKUs ({missing_hs_pct:.2f}%) are missing UAE Customs HS Codes.")
print(f"Audit Alert: {missing_pg_count} SKUs ({missing_pg_pct:.2f}%) are missing Gen. Prod. Posting Groups.")
print("\n Action Required: These anomalies will trigger exceptions in our pre-distribution master data gatekeeper module!")

--- Item Master Missing Values Audit ---
HS_Code                   30
Gen_Prod_Posting_Group    49
dtype: int64

Audit Alert: 30 SKUs (16.67%) are missing UAE Customs HS Codes.
Audit Alert: 49 SKUs (27.22%) are missing Gen. Prod. Posting Groups.

 Action Required: These anomalies will trigger exceptions in our pre-distribution master data gatekeeper module!


#### **DuckDB In-Memory SQL Integration Test**

In [7]:
# Initialize in-memory DuckDB connection
con = duckdb.connect(database=":memory:")

# Register raw CSV extracts as virtual SQL tables
con.execute(f"""
    CREATE OR REPLACE TABLE sales AS SELECT * FROM read_csv_auto('{DATA_PATH}/d365_sales_lines.csv');
    CREATE OR REPLACE TABLE invoices AS SELECT * FROM read_csv_auto('{DATA_PATH}/carrier_freight_invoices.csv');
    CREATE OR REPLACE TABLE items AS SELECT * FROM read_csv_auto('{DATA_PATH}/d365_item_master.csv');
""")

# Execute SQL query: Carrier invoice financial variance and overcharge summary
carrier_variance_df = con.execute("""
    SELECT 
        i.Carrier_Code,
        COUNT(i.Invoice_No) AS Total_Invoices,
        SUM(CASE WHEN i.Anomaly_Flag THEN 1 ELSE 0 END) AS Anomaly_Count,
        ROUND(SUM(i.Variance_AED), 2) AS Total_Variance_AED,
        ROUND(AVG(i.Variance_AED), 2) AS Avg_Variance_AED
    FROM invoices i
    GROUP BY i.Carrier_Code
    ORDER BY Total_Variance_AED DESC;
""").df()

print("--- Carrier Invoice Financial Variance Summary ---")
display(carrier_variance_df)

--- Carrier Invoice Financial Variance Summary ---


,Carrier_Code,Total_Invoices,Anomaly_Count,Total_Variance_AED,Avg_Variance_AED
0,DHL Express,339,55.0,217692.17,642.16
1,Aramex GCC,351,63.0,209668.97,597.35
2,Maersk Logistics,310,51.0,118767.82,383.12
